# DuckDB Smoke Test

Verifies DuckDB is operational in the Python kernel inside the container.
Run via: `docker compose run --rm jupyter papermill notebooks/test_duckdb.ipynb notebooks/test_duckdb.ipynb`

In [1]:
import duckdb
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.5.1


In [2]:
import pandas as pd

df = pd.DataFrame({
    "region": ["EU", "US", "EU", "APAC", "US", "APAC"],
    "sales":  [1200, 3400, 980, 2100, 4500, 1750],
    "units":  [12, 34, 10, 21, 45, 17],
})

result = duckdb.sql("""
    SELECT
        region,
        COUNT(*)        AS rows,
        SUM(sales)      AS total_sales,
        AVG(sales)      AS avg_sales,
        SUM(units)      AS total_units
    FROM df
    GROUP BY region
    ORDER BY total_sales DESC
""").df()

print(result.to_string(index=False))

region  rows  total_sales  avg_sales  total_units
    US     2       7900.0     3950.0         79.0
  APAC     2       3850.0     1925.0         38.0
    EU     2       2180.0     1090.0         22.0


In [3]:
import os, tempfile

csv_path = "/tmp/doml_smoke_test.csv"
df.to_csv(csv_path, index=False)

result2 = duckdb.sql(f"""
    SELECT region, SUM(sales) AS total_sales
    FROM '{csv_path}'
    GROUP BY region
    ORDER BY total_sales DESC
""").df()

print(result2.to_string(index=False))
os.remove(csv_path)
print("PASS: DuckDB CSV scan operational")

region  total_sales
    US       7900.0
  APAC       3850.0
    EU       2180.0
PASS: DuckDB CSV scan operational


In [4]:
project_root = os.environ.get("PROJECT_ROOT", "NOT SET")
print(f"PROJECT_ROOT = {project_root}")
assert project_root != "NOT SET", "PROJECT_ROOT env var is not set inside container"
print("PASS: PROJECT_ROOT env var present")

PROJECT_ROOT = /home/jovyan/work
PASS: PROJECT_ROOT env var present
